# ACE-Step Premium on Google Colab (Tesla T4)

This notebook installs and runs **ACE-Step Premium v1** on a free Colab Tesla T4 GPU and exposes the Gradio UI through an **ngrok** public URL.

**Before you start**
1. `Runtime` -> `Change runtime type` -> Hardware accelerator = **GPU**, GPU type = **T4**.
2. Make a free account at https://dashboard.ngrok.com and copy your **Authtoken** from `Your Authtoken`.
3. Paste that token into the cell marked **NGROK TOKEN** further down.

Run cells **in order, top to bottom**. The model download cell (Step 5) is the slowest -- give it 10-25 minutes.

## Step 1 - Confirm the GPU is a Tesla T4

In [ ]:
!nvidia-smi
import torch
print('CUDA available :', torch.cuda.is_available())
print('Device name    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('Torch version  :', torch.__version__)
print('Torch CUDA     :', torch.version.cuda)
assert torch.cuda.is_available(), 'No GPU detected. Set Runtime -> Change runtime type -> T4 GPU.'

## Step 2 - Install a recent FFmpeg
ACE-Step needs FFmpeg to encode WAV / MP3 / M4A. The Colab default is older; we drop in a current static build.

In [ ]:
%cd /content
!wget -q https://huggingface.co/MonsterMMORPG/Wan_GGUF/resolve/main/ffmpeg-N-121105-ga0936b9769-linux64-gpl.tar.xz
!tar xf ffmpeg-N-121105-ga0936b9769-linux64-gpl.tar.xz
!mv -f ffmpeg-N-121105-ga0936b9769-linux64-gpl/bin/ffmpeg  /usr/local/bin/
!mv -f ffmpeg-N-121105-ga0936b9769-linux64-gpl/bin/ffprobe /usr/local/bin/
!chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe
!ffmpeg -version | head -n 1

## Step 3 - Clone the ACE-Step Premium repository

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/FurkanGozukara/ACE-Step_Premium
%cd /content/ACE-Step_Premium
!git reset --hard
!git pull

## Step 4 - Install Python dependencies (Tesla T4 compatible)

The original `requirements_ACE_Step.txt` ships wheels for **CUDA 13.1 / Torch 2.9.1** (flash_attn, xformers, sageattention).
Colab T4 runs CUDA 12.x and is Turing (SM 7.5), so those wheels will not install or will not run.
We use a stripped-down requirements file that relies on Colab's pre-installed PyTorch and falls back
to PyTorch's built-in scaled-dot-product attention.

We also build `nano-vllm` from the cloned source tree.

In [ ]:
%%writefile /content/requirements_ACE_Step_Colab_T4.txt
huggingface-hub
hf_xet
requests>=2.31.0
gradio==6.10.0
transformers>=4.51.0,<4.58.0
diffusers
matplotlib>=3.7.5
scipy>=1.10.1
pytorch_wavelets
PyWavelets
soundfile>=0.13.1
loguru>=0.7.3
einops>=0.8.1
accelerate>=1.12.0
fastapi>=0.110.0
uvicorn[standard]>=0.27.0
numba>=0.63.1
vector-quantize-pytorch>=1.27.15
torchao==0.15.0
toml
modelscope
peft>=0.18.0
lycoris-lora
lightning>=2.0.0
tensorboard>=2.0.0
xxhash
safetensors==0.7.0

In [ ]:
# Install the slimmed requirements
!pip install -q --upgrade pip
!pip install -q -r /content/requirements_ACE_Step_Colab_T4.txt

In [ ]:
# Build & install nano-vllm from the cloned source
!pip install -q /content/ACE-Step_Premium/acestep/third_parts/nano-vllm

## Step 5 - Download the ACE-Step models (XL Turbo bundle)

Tesla T4 has 16 GB VRAM, so we download **only the XL Turbo** DiT plus the shared components
(VAE, Qwen3 embedding, 5Hz LMs). This is the lightest premium config and runs comfortably on T4.

If a download stalls or fails, just re-run the cell -- the downloader resumes and verifies SHA256.

In [ ]:
import os
os.environ['ACESTEP_PROJECT_ROOT']      = '/content/ACE-Step_Premium'
os.environ['ACESTEP_CHECKPOINTS_DIR']   = '/content/ACE-Step_Premium/models'
os.environ['HF_HOME']                   = '/content/hf_cache'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

import urllib.request
if not os.path.isfile('/content/ACE-Step_Premium/HF_model_downloader.py'):
    print('Downloader missing from repo - fetching raw copy from GitHub')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/kej83/prog-1imb-25-26/refs/heads/main/Ai-stuff/Ai-sound-gen/HF_model_downloader.py',
        '/content/ACE-Step_Premium/HF_model_downloader.py')

%cd /content/ACE-Step_Premium
!python /content/ACE-Step_Premium/HF_model_downloader.py --model acestep-v15-xl-turbo \
  || python /content/ACE-Step_Premium/HF_model_downloader.py --model acestep-v15-xl-turbo

## Step 6 - Install pyngrok and paste your ngrok token

Get your token from https://dashboard.ngrok.com/get-started/your-authtoken and paste it into the string below.

In [ ]:
!pip install -q pyngrok

# ---------- NGROK TOKEN ----------
NGROK_AUTHTOKEN = 'PASTE_YOUR_NGROK_AUTHTOKEN_HERE'
# ----------------------------------

from pyngrok import ngrok, conf
assert NGROK_AUTHTOKEN and NGROK_AUTHTOKEN != 'PASTE_YOUR_NGROK_AUTHTOKEN_HERE', \
    'Paste your ngrok authtoken into NGROK_AUTHTOKEN first.'
ngrok.set_auth_token(NGROK_AUTHTOKEN)
print('ngrok configured.')

## Step 7 - Open the tunnel and launch the app

Gradio defaults to port **7860**. We open an ngrok tunnel to that port first, print the public URL,
then start the app. **Open the printed URL in your browser** -- that is the UI you share with your students.

Leave this cell running. Stop it (square button) to shut the app down.

In [ ]:
import os
from pyngrok import ngrok

# Close any leftover tunnels from previous attempts
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public = ngrok.connect(7860, 'http')
print('=' * 70)
print('  ACE-Step Premium PUBLIC URL:')
print('  ', public.public_url)
print('=' * 70)

os.environ['ACESTEP_PROJECT_ROOT']      = '/content/ACE-Step_Premium'
os.environ['ACESTEP_CHECKPOINTS_DIR']   = '/content/ACE-Step_Premium/models'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['PYTHONWARNINGS']            = 'ignore'
os.environ['GRADIO_SERVER_NAME']        = '0.0.0.0'
os.environ['GRADIO_SERVER_PORT']        = '7860'

os.chdir('/content/ACE-Step_Premium')
# Launch the Gradio UI. Streams output live into this cell.
!python -m acestep.acestep_v15_pipeline

## Optional - Stop the app cleanly

Use the square (Stop) button on Step 7 first, then run this cell to free port 7860 and tear down the tunnel:

In [ ]:
from pyngrok import ngrok
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
ngrok.kill()
!fuser -k 7860/tcp 2>/dev/null || true
print('Tunnel closed and port 7860 released.')

## Troubleshooting (T4 specifics)

**`CUDA out of memory`**
Tesla T4 has 16 GB VRAM. Stick to the **XL Turbo** model (Step 5 already downloads only that one).
In the UI, keep generation duration around 60 s and use the 0.6 B or 1.7 B LM rather than the 4 B.

**ngrok says 'failed to complete tunnel connection'**
Wait until Step 7 prints `Running on local URL: http://0.0.0.0:7860` before opening the URL.
Gradio needs roughly 30-90 s after launch to load the models.

**`flash_attn` / `sageattention` import errors**
Expected on T4 -- we intentionally do not install them. The app must fall back to PyTorch SDP attention.
If a code path still imports them and crashes, set `os.environ['ACESTEP_DISABLE_FLASH']='1'` before Step 7 and re-run.

**Session disconnected**
Free Colab kills idle sessions after about 90 minutes. Re-run Steps 6 and 7 to get a new public URL;
models stay in `/content` for the rest of the session and skip on re-download.